In [1]:
using JuMP
using HiGHS
# using EzXML
using Plots
# using Ipopt
# using Cbc
import MultiObjectiveAlgorithms as MOA
import MathOptInterface as MOI
using Printf
using PrettyTables

In [2]:
function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
	Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
	Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
	Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
	Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
	c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
	h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
	w = parse.(Int, split(lines[idx])); idx += 1
	b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
	c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
	v = parse.(Int, split(lines[idx])); idx += 1
	V = parse.(Int, split(lines[idx])); idx += 1

	B_star = Array{Int, 3}(undef, S, P, length(D_star))
	for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

	C_star = Array{Int, 3}(undef, T, P, length(D_star))
	for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

	if validate==true
		# input validation


	end

	return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star

end

read_input (generic function with 1 method)

In [3]:
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7

7

In [4]:
file_path = "input/input_example_january.txt"
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

(14, [1, 3, 5, 8, 10, 12], [7, 14], 5, SubString{String}["Player1", "Player2", "Player3", "Player4", "Player5"], 3, SubString{String}["Skill1", "Skill2", "Skill3"], 3, SubString{String}["Tactic1", "Tactic2", "Tactic3"], 10, SubString{String}["Exercise1", "Exercise2", "Exercise3", "Exercise4", "Exercise5", "Exercise6", "Exercise7", "Exercise8", "Exercise9", "Exercise10"], [100, 90, 110, 85, 100], [[5, 6, 7, 8, 6], [8, 9, 5, 6, 7], [4, 5, 9, 8, 6]], [[7, 6, 4, 6, 6], [8, 5, 5, 8, 7], [4, 5, 4, 8, 6]], [[1, 1, 1, 1, 1, 1], [1, 1, 0, 1, 1, 0], [1, 0, 1, 1, 1, 1], [1, 0, 1, 1, 0, 1], [1, 1, 1, 1, 1, 1]], [1, 3, 4, 5, 2, 1, 5, 4, 2, 3], [[5, 5, 4, 3, 3, 3, 4, 3, 3, 5], [4, 3, 2, 5, 1, 5, 5, 3, 4, 4], [1, 2, 3, 4, 5, 1, 2, 3, 4, 5]], [[5, 5, 4, 3, 3, 3, 4, 3, 3, 5], [4, 3, 2, 5, 1, 5, 5, 3, 4, 4], [1, 2, 3, 4, 5, 1, 2, 3, 4, 5]], [2, 3, 4, 2, 1, 1, 3, 4, 3, 1], [18, 18, 18, 18, 18, 18], [10 10 … 10 10; 9 9 … 9 9; 8 8 … 8 8;;; 20 20 … 20 20; 18 18 … 18 18; 16 16 … 16 16], [10 10 … 10 10; 9 9 …

In [5]:
function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

ρ (generic function with 1 method)

In [6]:
# model = Model(() -> MOA.Optimizer(HiGHS.Optimizer))
model = Model(() -> MOA.Optimizer(
    optimizer_with_attributes(
		HiGHS.Optimizer, 
		"output_flag" => true,
        "mip_rel_gap" => 0.05,
        "threads" => 8
	))
)
# model = Model(optimizer_with_attributes(HiGHS.Optimizer, "threads" => Threads.nthreads()))
# model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: MOA[algorithm=MultiObjectiveAlgorithms.Lexicographic, optimizer=HiGHS]
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [7]:
# Переменные

@variable(model, x[1:E, 1:length(D_)] >= 0, Int)

10×6 Matrix{VariableRef}:
 x[1,1]   x[1,2]   x[1,3]   x[1,4]   x[1,5]   x[1,6]
 x[2,1]   x[2,2]   x[2,3]   x[2,4]   x[2,5]   x[2,6]
 x[3,1]   x[3,2]   x[3,3]   x[3,4]   x[3,5]   x[3,6]
 x[4,1]   x[4,2]   x[4,3]   x[4,4]   x[4,5]   x[4,6]
 x[5,1]   x[5,2]   x[5,3]   x[5,4]   x[5,5]   x[5,6]
 x[6,1]   x[6,2]   x[6,3]   x[6,4]   x[6,5]   x[6,6]
 x[7,1]   x[7,2]   x[7,3]   x[7,4]   x[7,5]   x[7,6]
 x[8,1]   x[8,2]   x[8,3]   x[8,4]   x[8,5]   x[8,6]
 x[9,1]   x[9,2]   x[9,3]   x[9,4]   x[9,5]   x[9,6]
 x[10,1]  x[10,2]  x[10,3]  x[10,4]  x[10,5]  x[10,6]

In [8]:
@variable(model, y[1:S, 1:P, 1:length(D_star)], Bin)

3×5×2 Array{VariableRef, 3}:
[:, :, 1] =
 y[1,1,1]  y[1,2,1]  y[1,3,1]  y[1,4,1]  y[1,5,1]
 y[2,1,1]  y[2,2,1]  y[2,3,1]  y[2,4,1]  y[2,5,1]
 y[3,1,1]  y[3,2,1]  y[3,3,1]  y[3,4,1]  y[3,5,1]

[:, :, 2] =
 y[1,1,2]  y[1,2,2]  y[1,3,2]  y[1,4,2]  y[1,5,2]
 y[2,1,2]  y[2,2,2]  y[2,3,2]  y[2,4,2]  y[2,5,2]
 y[3,1,2]  y[3,2,2]  y[3,3,2]  y[3,4,2]  y[3,5,2]

In [9]:
@variable(model, z[1:T, 1:P, 1:length(D_star)], Bin)

3×5×2 Array{VariableRef, 3}:
[:, :, 1] =
 z[1,1,1]  z[1,2,1]  z[1,3,1]  z[1,4,1]  z[1,5,1]
 z[2,1,1]  z[2,2,1]  z[2,3,1]  z[2,4,1]  z[2,5,1]
 z[3,1,1]  z[3,2,1]  z[3,3,1]  z[3,4,1]  z[3,5,1]

[:, :, 2] =
 z[1,1,2]  z[1,2,2]  z[1,3,2]  z[1,4,2]  z[1,5,2]
 z[2,1,2]  z[2,2,2]  z[2,3,2]  z[2,4,2]  z[2,5,2]
 z[3,1,2]  z[3,2,2]  z[3,3,2]  z[3,4,2]  z[3,5,2]

In [10]:
@variable(model, Z[1:T, 1:length(D_star)], Bin)

3×2 Matrix{VariableRef}:
 Z[1,1]  Z[1,2]
 Z[2,1]  Z[2,2]
 Z[3,1]  Z[3,2]

In [11]:
@variable(model, A_min[1:length(D_star)])

2-element Vector{VariableRef}:
 A_min[1]
 A_min[2]

In [12]:
function A(p::Int, d::Int)
	result = a0[p]
	result += sum(δ1(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	result -= sum(δ2(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function A_avg(d::Int)
	return sum(A(p, d) for p in 1:P)/P
end

α = 0.5
function A()
	return sum(α*A_min[d_]+(1-α)*A_avg(D_star[d_]) for d_ in 1:length(D_star))
end

function B(s::Int, p::Int, d::Int)
	result = b0[s][p]
	result += sum(h[p][d_]*sum(b[s][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function B()
	return sum(sum(sum(β(s, p, d_)*y[s, p, d_] for s in 1:S) for p in 1:P) for d_ in 1:length(D_star))
end

function C(t::Int, p::Int, d::Int)
	result = c0[t][p]
	result += sum(h[p][d_]*sum(c[t][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
end

function C()
	return sum(sum(γ(t, d_)*Z[t, d_] for t in 1:T) for d_ in 1:length(D_star))
end

C (generic function with 2 methods)

In [ ]:
# Ограничения
@constraint(model, [d_=1:length(D_)], sum(v[e]*x[e, d_] for e in 1:E) <= V[d_])

6-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 2 x[1,1] + 3 x[2,1] + 4 x[3,1] + 2 x[4,1] + x[5,1] + x[6,1] + 3 x[7,1] + 4 x[8,1] + 3 x[9,1] + x[10,1] <= 18
 2 x[1,2] + 3 x[2,2] + 4 x[3,2] + 2 x[4,2] + x[5,2] + x[6,2] + 3 x[7,2] + 4 x[8,2] + 3 x[9,2] + x[10,2] <= 18
 2 x[1,3] + 3 x[2,3] + 4 x[3,3] + 2 x[4,3] + x[5,3] + x[6,3] + 3 x[7,3] + 4 x[8,3] + 3 x[9,3] + x[10,3] <= 18
 2 x[1,4] + 3 x[2,4] + 4 x[3,4] + 2 x[4,4] + x[5,4] + x[6,4] + 3 x[7,4] + 4 x[8,4] + 3 x[9,4] + x[10,4] <= 18
 2 x[1,5] + 3 x[2,5] + 4 x[3,5] + 2 x[4,5] + x[5,5] + x[6,5] + 3 x[7,5] + 4 x[8,5] + 3 x[9,5] + x[10,5] <= 18
 2 x[1,6] + 3 x[2,6] + 4 x[3,6] + 2 x[4,6] + x[5,6] + x[6,6] + 3 x[7,6] + 4 x[8,6] + 3 x[9,6] + x[10,6] <= 18

In [14]:
@constraint(model, [s=1:S, p=1:P, d_=1:length(D_star)], B(s, p, D_star[d_]) + B_star[s, p, d_]*(1-y[s, p, d_]) >= B_star[s, p, d_])

3×5×2 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 3 x[9,1] + 5 x[10,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 3 x[9,2] + 5 x[10,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 3 x[9,3] + 5 x[10,3] - 10 y[1,1,1] >= -5  …  5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 3 x[9,1] + 5 x[10,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 3 x[9,2] + 5 x[10,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 3 x[9,3] + 5 x[10,3] - 10 y[1,5,1] >= -6
 4 x[1,1] + 3 x[2,1] + 2 x[3,1] + 5 x[4,1] + x[5,1] + 5 x[6,1] + 5 x[7,1] + 3 x[8,1] + 4 x[9,1] + 4 x[10,1] +

In [15]:
@constraint(model, [t=1:T, p=1:P, d_=1:length(D_star)], C(t, p, D_star[d_]) + C_star[t, p, d_]*(1-z[t, p, d_]) >= C_star[t, p, d_])

3×5×2 Array{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}, 3}:
[:, :, 1] =
 5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 3 x[9,1] + 5 x[10,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 3 x[9,2] + 5 x[10,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 3 x[9,3] + 5 x[10,3] - 10 z[1,1,1] >= -7  …  5 x[1,1] + 5 x[2,1] + 4 x[3,1] + 3 x[4,1] + 3 x[5,1] + 3 x[6,1] + 4 x[7,1] + 3 x[8,1] + 3 x[9,1] + 5 x[10,1] + 5 x[1,2] + 5 x[2,2] + 4 x[3,2] + 3 x[4,2] + 3 x[5,2] + 3 x[6,2] + 4 x[7,2] + 3 x[8,2] + 3 x[9,2] + 5 x[10,2] + 5 x[1,3] + 5 x[2,3] + 4 x[3,3] + 3 x[4,3] + 3 x[5,3] + 3 x[6,3] + 4 x[7,3] + 3 x[8,3] + 3 x[9,3] + 5 x[10,3] - 10 z[1,5,1] >= -6
 4 x[1,1] + 3 x[2,1] + 2 x[3,1] + 5 x[4,1] + x[5,1] + 5 x[6,1] + 5 x[7,1] + 3 x[8,1] + 4 x[9,1] + 4 x[10,1] +

In [16]:
@constraint(model, [t=1:T, d_=1:length(D_star)], sum(z[t, p, d_] for p in 1:P) >= ρ(t)*Z[t, d_])

3×2 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}}:
 z[1,1,1] + z[1,2,1] + z[1,3,1] + z[1,4,1] + z[1,5,1] - 2 Z[1,1] >= 0  …  z[1,1,2] + z[1,2,2] + z[1,3,2] + z[1,4,2] + z[1,5,2] - 2 Z[1,2] >= 0
 z[2,1,1] + z[2,2,1] + z[2,3,1] + z[2,4,1] + z[2,5,1] - 2 Z[2,1] >= 0     z[2,1,2] + z[2,2,2] + z[2,3,2] + z[2,4,2] + z[2,5,2] - 2 Z[2,2] >= 0
 z[3,1,1] + z[3,2,1] + z[3,3,1] + z[3,4,1] + z[3,5,1] - 2 Z[3,1] >= 0     z[3,1,2] + z[3,2,2] + z[3,3,2] + z[3,4,2] + z[3,5,2] - 2 Z[3,2] >= 0

In [17]:
@constraint(model, [d_=1:length(D_star), p=1:P], A_min[d_] <= A(p, D_star[d_]))

2×5 Matrix{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.LessThan{Float64}}, ScalarShape}}:
 -0.0181322083962816 x[1,1] - 0.054396625188844805 x[2,1] - 0.0725288335851264 x[3,1] - 0.09066104198140845 x[4,1] - 0.0362644167925632 x[5,1] - 0.0181322083962816 x[6,1] - 0.09066104198140845 x[7,1] - 0.0725288335851264 x[8,1] - 0.0362644167925632 x[9,1] - 0.054396625188844805 x[10,1] + 0.22027980113880552 x[1,2] + 0.6608394034164164 x[2,2] + 0.8811192045552221 x[3,2] + 1.1013990056940282 x[4,2] + 0.44055960227761104 x[5,2] + 0.22027980113880552 x[6,2] + 1.1013990056940282 x[7,2] + 0.8811192045552221 x[8,2] + 0.44055960227761104 x[9,2] + 0.6608394034164164 x[10,2] + 0.5494576313170952 x[1,3] + 1.6483728939512852 x[2,3] + 2.197830525268381 x[3,3] + 2.7472881565854763 x[4,3] + 1.0989152626341905 x[5,3] + 0.5494576313170952 x[6,3] + 2.7472881565854763 x[7,3] + 2.197830525268381 x[8,3] + 1.0989152626341905 x[9,3] + 1.648372893

In [ ]:
# @objective(model, Max, A() + B() + C())
@objective(model, Max, [A(), B(), C()])

# set_optimizer_attribute(model, "mip_rel_gap", 0.05)

# set_time_limit_sec(model, 10.0)
set_attribute(model, MOI.TimeLimitSec(), 5.0)

set_attribute(model, MOA.Algorithm(), MOA.Hierarchical())
set_attribute(model, MOA.ComputeIdealPoint(), true)

# set_attribute(model, MOA.SolutionLimit(), 1)

unset_silent(model)
# set_log_file(model, "solver_log.txt")
open("highs_moa.log", "w") do io
    redirect_stdout(io) do
        optimize!(model)
    end
end

# set_attribute(model, "output_flag", true)
# set_attribute(model, MOA.OptimizerAttribute("output_flag"), true)

# optimize!(model)

In [ ]:
if termination_status(model) == TIME_LIMIT
    if has_values(model)
        println("Тайм-аут: Найдено решение с точностью ", relative_gap(model))
    else
        println("Тайм-аут: Решений не найдено вообще.")
    end
end

In [ ]:
solution_summary(model)

In [ ]:
println("Ideal Point (Bound): ", objective_bound(model))

In [ ]:
println("Found ", result_count(model), " solutions.")

for i in 1:result_count(model)
    println("Solution #$i:")
    println("  x: ", value.(x; result=i))
    println("  Objectives: ", objective_value(model; result = i))
end

In [ ]:
Days = length(D_star) 
plots_list = []

for d in 1:Days
	data_slice = value(y[1:S, 1:P, d])
	p = heatmap(
		data_slice,
		size = (800, 800), 
		xlims = (1, P),
		ylims = (1, S),
		title = "Освоенные навыки к к.д. №$(d)",
		xticks = (1:P), 
		yticks = (1:S),
		colormap = :binary, 
		xlabel = "Игроки", 
		ylabel = "Навыки",
		aspect_ratio = :equal,
		colorbar = false
	)
	push!(plots_list, p)
end

final_plot = plot(plots_list..., layout = (1, Days), size = (1600 * Days, 1600), plot_title = "Освоение навыков по дням")
savefig(final_plot, "skills_plot.png")

In [ ]:
Days = length(D_star) 
plots_list = []

for d in 1:Days
	data_slice = value(z[1:T, 1:P, d])
	p = heatmap(
		data_slice,
		size = (800, 800), 
		xlims = (1, P),
		ylims = (1, T),
		title = "Освоенные индивидуально тактики к к.д. №$(d)",
		xticks = (1:P), 
		yticks = (1:T),
		colormap = :binary, 
		xlabel = "Игроки", 
		ylabel = "Тактики",
		aspect_ratio = :equal,
		colorbar = false
	)
	push!(plots_list, p)
end

final_plot = plot(plots_list..., layout = (1, Days), size = (1600 * Days, 1600), plot_title = "Освоенные индивидуально тактики по дням")
savefig(final_plot, "tactics_individual_plot.png")

In [ ]:
Days = length(D_star) 

data_slice = value(Z[1:T, 1:Days])
p = heatmap(
	data_slice,
	size = (800, 800), 
	xlims = (1, T),
	ylims = (1, Days),
	title = "Освоенные командой тактики по к.д.",
	xticks = (1:T), 
	yticks = (1:Days),
	colormap = :binary, 
	xlabel = "Тактики", 
	ylabel = "Контрольные даты",
	aspect_ratio = :equal,
	colorbar = false
)

savefig(p, "tactics_team_plot.png")

In [ ]:
println("---Итоговая эффективность тренировок---")
println("A - общая физическая готовность команды\nB - бонус за освоенные индивидуальные навыки\nC - бонус за освоенные командой тактики\n")
println("A = $(value(A()))\nB = $(value(B()))\nC = $(value(C()))\n\n")

println("---Расписание тренировок---\n")
for d_ in 1:length(D_)
	println("Тренировка №$(d_) (день №$(D_[d_]))")
	for e in 1:E
		if value(x[e, d_]) >= 1
			print("$(Exercise_names[e]) ")
		end
	end
	print("\n\n")
end

p1 = plot(title="тренированность игроков по дням", xlabel="День", ylabel="Тренированность")
for p in 1:P
	A_values = [value(A(p, d)) for d in 1:D]
	plot!(p1, 1:D, A_values, label="$(Player_names[p])")
end
vline!(D_, label="Дни тренировок")
vline!(D_star, label="Контрольные дни")

display(p1)

println("---Освоенные навыки---\n")
for p in 1:P
	for d_ in 1:length(D_star)	
		println("Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
		learned_anything = false
		for s in 1:S
			if (value(y[s, p, d_]) == 1 && (d_ == 1 || value(y[s, p, d_-1]) != 1.0))
				learned_anything = true
				println("Освоил навык '$(Skill_names[s])'")
			end
		end
		if (learned_anything == false)
			println("Ничего не освоил")
		end
	end
	print("\n")
end

println("---Освоенные индивидуально тактики---\n")
for p in 1:P
	for d_ in 1:length(D_star)	
		println("Игрок: $(Player_names[p]), контрольная дата №$(d_) (день $(D_star[d_]))")
		learned_anything = false
		for t in 1:T
			if (value(z[t, p, d_]) == 1 && (d_ == 1 || value(z[t, p, d_-1]) != 1.0))
				learned_anything = true
				println("Освоил тактику '$(Tactic_names[t])'")
			end
		end
		if (learned_anything == false)
			println("Ничего не освоил")
		end
	end
	print("\n")
end

println("---Освоенные командой тактики---\n")
for d_ in 1:length(D_star)	
	for t in 1:T	
		println("Тактика: $(Tactic_names[t]), контрольная дата №$(d_) (день $(D_star[d_]))")
		learned_anything = false
		if value(Z[t, d_]) == 1
			println("Освоено")
		else
			println("Не освоено")
		end
	end
	print("\n")
end

In [ ]:
println("---Расписание тренировок---\n")

n_days = length(D_)

# Таблица: каждая строка — один день
data = Array{Any}(undef, n_days, 2)

for d_ in 1:n_days
    # 1-й столбец: информация о дне/тренировке
    data[d_, 1] = "Тренировка №$(d_) (день №$(D_[d_]))"

    # Собираем все упражнения, попавшие в этот день
    day_exercises = String[]
    for e in 1:E
        if value(x[e, d_]) >= 1
            push!(day_exercises, Exercise_names[e])
        end
    end

    # 2-й столбец: список упражнений через запятую
    data[d_, 2] = join(day_exercises, ", ")
end

header = ["День", "Упражнения"]

# ВАЖНО: header — позиционный аргумент, без keyword-параметров
pretty_table(data)